# MedGemma T1 Evaluation

This notebook evaluates the recovered T1 LoRA adapter on the split contained in the uploaded bundle. It does not retrain the model. Select a T4 or L4 GPU and enable internet access before running.

In [ ]:
%pip install -q -U "transformers>=4.50.0" peft accelerate bitsandbytes pillow scikit-learn safetensors
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import tarfile

print("Upload colab-t1-eval-bundle.tar.gz or colab-t1-test-bundle.tar.gz")
uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith(".tar.gz"))
expected_records = 477 if "test" in bundle_name else 488
bundle_path = Path("/content") / bundle_name
root = Path("/content/eval_bundle")
root.mkdir(exist_ok=True)
with tarfile.open(bundle_path, "r:gz") as archive:
    archive.extractall(root)
print("Bundle root:", root)
print("Adapter archive:", root / "medgemma-t1-qlora-adapter.zip")

In [ ]:
import os
import getpass
import zipfile

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste HF_TOKEN: ")

adapter_zip = root / "medgemma-t1-qlora-adapter.zip"
with zipfile.ZipFile(adapter_zip) as archive:
    archive.extractall(root)
adapter_dir = root / "medgemma-t1-qlora"
assert (adapter_dir / "adapter_model.safetensors").exists()
assert (root / "data/secondary/instruction_tuning_mixed/val.jsonl").exists()
print("Adapter ready:", adapter_dir)

In [ ]:
import subprocess
import sys

runner = root / "scripts/train_t1_qlora.py"
command = [
    sys.executable, "-u", str(runner),
    "--root", str(root),
    "--output", str(adapter_dir),
    "--eval-only",
    "--max-val", str(expected_records),
    "--max-new-tokens", "8",
]
subprocess.run(command, check=True)

In [ ]:
import json
import shutil
from google.colab import files

metrics_path = adapter_dir / "t1_metrics.json"
predictions_path = adapter_dir / "t1_predictions.jsonl"
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))
assert metrics["records"] == expected_records
split_name = "test" if expected_records == 477 else "validation"
results_zip = shutil.make_archive(f"/content/t1-{split_name}-results", "zip", root_dir=adapter_dir.parent, base_dir=adapter_dir.name)
print("Download:", results_zip)
files.download(results_zip)